# CHLA Weekly
* BGC-Argo profiles with CHLA are indicated by black circles.
* 8-day composite GCOM-C (Shikisai) CHLA

In [1]:
import plotly.io as pio
# This forces Plotly to embed the interactive JavaScript engine
# even when the notebook is run by a background bot!
pio.renderers.default = 'notebook_connected'

import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import plotly.graph_objects as go
import plotly.express as px
import sys
import os
import urllib.request
import urllib.error
import re
import xarray as xr

# 1. Get the absolute path to the 'lessons' folder 
# ('..' means "go up one level to the root directory")
lessons_path = os.path.abspath(os.path.join('..', 'lessons'))

# 2. Add that folder to Python's search path
if lessons_path not in sys.path:
    sys.path.append(lessons_path)

from ipynb.fs.defs.lesson_1 import load_index, filter_index

In [2]:
def download_latest_gcom_c_daytime():
    """Scrapes the JAXA open directory to find and download the latest Daytime (D) CHLA data."""
    
    # 1. Start with the current year
    year = datetime.now().year
    
    for attempt in range(2): 
        base_url = f"https://kuroshio.eorc.jaxa.jp/pub/SGLI_STD/Global_05km/8-day/CHLA/{year}/"
        print(f"Checking directory: {base_url}")
        
        try:
            # 2. Read the raw HTML of the directory webpage
            req = urllib.request.urlopen(base_url, timeout=10)
            html = req.read().decode('utf-8')
            
            # 3. FIX: Find ONLY files with 'D08D' (Descending/Daytime) right after the 8-digit date
            filenames = re.findall(r'GC1SG1_\d{8}D08D_[\w_]+\.nc', html)
            
            if filenames:
                # 4. Remove duplicates and sort alphabetically by date.
                filenames = sorted(list(set(filenames)))
                latest_file = filenames[-1] # Grab the newest Daytime file
                
                download_url = base_url + latest_file
                print(f"⭐ Found latest daytime data: {latest_file}")
                
                # 5. Download the file
                print("Downloading... (this may take a moment)")
                urllib.request.urlretrieve(download_url, latest_file)
                print("✅ Download complete!")
                # Slice the string from index 7 up to (but not including) 15
                latest_date = latest_file[7:15]
                
                return latest_file,latest_date
            
            else:
                print(f"No Daytime .nc files found in the {year} folder.")
                
        except urllib.error.URLError:
            print(f"Folder for {year} does not exist yet.")
            
        # Fallback to the previous year
        year -= 1
        print("Falling back to previous year's folder...\n")
        
    raise FileNotFoundError("Could not find any GCOM-C Daytime files on the server.")

In [3]:
# Run the function!
latest_file,latest_date = download_latest_gcom_c_daytime()

Checking directory: https://kuroshio.eorc.jaxa.jp/pub/SGLI_STD/Global_05km/8-day/CHLA/2026/


⭐ Found latest daytime data: GC1SG1_20260415D08D_D0000_3MSG_CHLAM_3000.nc
Downloading... (this may take a moment)


✅ Download complete!


In [4]:
ds_jaxa = xr.open_dataset(latest_file)

In [5]:
lon0 = -180
lon1 = 180
lat0 = -90
lat1 = 90
date0 = latest_date
date1 = (datetime.strptime(latest_date, '%Y%m%d') + timedelta(days=7)).strftime('%Y%m%d')
bgc_parameters = ['CHLA']
bgc_mode = 'and'
df_index = load_index()
df_filter = filter_index(df_index, lon0, lon1, lat0, lat1, date0, date1, bgc_parameters, bgc_mode)


--- Connecting to the French GDAC ---


✅ Success! Data loaded from French GDAC.


In [6]:
def argo_shikisai_map(df_index, chla_ds):
    """
    Creates and displays an interactive 3D globe map and an interactive timeline
    directly inside the Jupyter Notebook.
    
    INPUT:
    * df_index: the filtered index file (pandas dataframe)
    * title_suffix (optional): string to append to the plot titles
    """
    
    # Pre-sort the entire DataFrame once
    df_index = df_index.sort_values(['wmo', 'datetime'])
    
    # --- 1. Figure Setup ---
    fig_map = go.Figure()
        
    # ==========================================
    # --- 1. SATELLITE LAYER (GCOM-C) ---
    # ==========================================
    
    # A. DOWNSAMPLE! Take every 10th pixel to prevent browser crash
    step = 10 
    
    # Extract arrays (assuming your xarray dataset uses these names)
    lats = chla_ds['latitude'].values[::step]
    lons = chla_ds['longitude'].values[::step]
    
    # Squeeze out extra dimensions (like time) if they exist, then slice
    chla = chla_ds['CHLA_AVE'].values.squeeze()[::step, ::step] 
    
    # B. Convert the 1D lat/lon axes into a 2D grid
    lon_grid, lat_grid = np.meshgrid(lons, lats)
    
    # C. Flatten everything into 1D lists so Plotly can read them as points
    lon_flat = lon_grid.flatten()
    lat_flat = lat_grid.flatten()
    chla_flat = chla.flatten()
    
    # D. Drop NaNs (landmasses and clouds) to save massive amounts of memory
    valid_mask = ~np.isnan(chla_flat)
    
    fig_map.add_trace(go.Scattergeo(
        lon=lon_flat[valid_mask],
        lat=lat_flat[valid_mask],
        mode='markers',
        marker=dict(
            size=1, # Tiny dots that blend together to look like a solid map
            symbol='square',
            color=chla_flat[valid_mask],
            colorscale='algae', # A great colorblind-friendly scale for CHLA
            colorbar=dict(title="mg/m3", x=0, len=0.6),
            showscale=True,
            cmin=0, # Set min/max to prevent outliers from washing out the colors
            cmax=10  
        ),
        name="GCOM-C CHLA",
        hoverinfo='skip' # Turn off hovering for these dots to speed up the map!
    ))    

    
    
    # --- 2. The Plotting Loop ---
    for i, (wmo, group) in enumerate(df_index.groupby('wmo')):
        
        label = f"{wmo}"
        
        # ------------------------------------
        # MAP: Plot Trajectory
        # ------------------------------------
        fig_map.add_trace(go.Scattergeo(
            lon=group['longitude'],
            lat=group['latitude'],
            mode='markers',
            marker=dict(size=4, color='black',symbol='circle-open'),
            name=label,
            # 1. Hand the datetime column to Plotly so it holds it in memory
            customdata=group['datetime'],   
            # 2. Use %{customdata} to display it, and format it nicely as a date!
            hovertemplate="<b>WMO:</b> " + str(wmo) + "<br><b>Date:</b> %{customdata|%Y-%m-%d %H:%M}<extra></extra>"
        ))
        
    # --- 3. Final Formatting ---
    
    # Format the Plotly Map (3D Globe)
    fig_map.update_geos(
        projection_type="orthographic",
#        showocean=True, oceancolor="LightBlue",
        showland=True, landcolor="LightGray",
        showcoastlines=True, coastlinecolor="DarkGray",
        # Explicitly turn on the grid for both axes:
        lonaxis_showgrid=True, lonaxis_gridcolor="White",
        lataxis_showgrid=True, lataxis_gridcolor="White"
    )
    
    # Rotate the globe so the data is front and center
    fig_map.update_layout(
        title=f"CHLA: {date0}-{date1}",
        showlegend=False,
        geo=dict(
            projection=dict(
                rotation=dict(
                    lon=140, 
                    lat=35,
                )
            )
        ),
        margin=dict(l=0, r=0, b=0, t=40),
        height=600 # Explicit height so it fills the notebook cell nicely
    )
    
    # --- 4. Display the plots ---
    fig_map.show()

In [7]:
argo_shikisai_map(df_filter,ds_jaxa)